# Suite PLC — Physical placement

`Placement` gives a diagram its coordinates, in diagram units, one mutation at a time, by translating the rules a mutation touches into linear constraints that an engine, Kiwi's, solves incrementally: each shape's points from its vectors' lengths and its unit length, its frame from its extent points and its grid, each grid from its children's sizes with collapsing margins, and the diagram the size of its grid. What it writes is a mutation, an upsert the store runs; each step asserts the frames and placements it wrote, read from the mutation, `[name, "frame", absolute, relative, size, unit_length]` or `[name, "placement", absolute, relative, direction]`, as Python's suite does, value for value. mbse-basis [CAP-242: incremental physical placement](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/README.md#CAP-242).

In [ ]:
import { assert, Bench, text } from "./support.js";
import { Constraints, Container, Kiwi, Logical, Placement } from "../src/Framework/index.js";

/** A square's four extent points, each a vector of length 2 from its origin: 2 units wide and high. */
const SQUARE: [string, number, number][] = [["n", 0, 2], ["s", 180, 2], ["e", 90, 2], ["w", 270, 2]];
const b = new Bench(true, true);
b.tune({ unit_length: 1, scaling_factor: 0.5, margin: 0.25 });

<a id="PLC-01"></a>
## PLC-01 · Each level's unit length: the diagram's, then its container's times the scaling factor

In [ ]:
b.child("", "cpu"); b.child("cpu", "alu");
assert(text([b.overview, b.shape("cpu"), b.shape("cpu.alu")].map((x) => Placement.unit_length(x))) === "[1,0.5,0.25]");
const plain = new Bench(true, true); plain.child("", "x");
assert(text(Placement.unit_length(plain.shape("x"))) === "1");

<a id="PLC-02"></a>
## PLC-02 · A shape's points lie its vectors' lengths times its unit length along their compasses; its frame bounds its extent points

In [ ]:
assert(text(b.points("cpu.alu", SQUARE)) === "\"[[\\\"overview\\\",\\\"frame\\\",[0,0],[0,0],[3,3.25],1],[\\\"overview/grid\\\",\\\"frame\\\",[0,0],[0,0],[3,3.25],1],[\\\"overview/grid/0,0\\\",\\\"frame\\\",[0.25,0.25],[0.25,0.25],[2.5,2.75],1],[\\\"cpu\\\",\\\"frame\\\",[0.5,0.5],[0.25,0.25],[2,2.25],0.5],[\\\"cpu/grid\\\",\\\"frame\\\",[0.5,0.75],[0,0.25],[2,2],0.5],[\\\"cpu/grid/0,0\\\",\\\"frame\\\",[0.75,1],[0.25,0.25],[1.5,1.5],0.5],[\\\"alu\\\",\\\"frame\\\",[1,1.25],[0.25,0.25],[1,1],0.25],[\\\"origin\\\",\\\"placement\\\",[1.5,1.75],[0.75,0.75],0],[\\\"n\\\",\\\"placement\\\",[1.5,1.25],[0,-0.5],0],[\\\"s\\\",\\\"placement\\\",[1.5,2.25],[0,0.5],180],[\\\"e\\\",\\\"placement\\\",[2,1.75],[0.5,0],90],[\\\"w\\\",\\\"placement\\\",[1,1.75],[-0.5,0],270]]\"");
assert(text(b.points("cpu", SQUARE)) === "\"[[\\\"origin\\\",\\\"placement\\\",[1.5,1.75],[1.25,1.5],0],[\\\"n\\\",\\\"placement\\\",[1.5,0.75],[0,-1],0],[\\\"s\\\",\\\"placement\\\",[1.5,2.75],[0,1],180],[\\\"e\\\",\\\"placement\\\",[2.5,1.75],[1,0],90],[\\\"w\\\",\\\"placement\\\",[0.5,1.75],[-1,0],270]]\"");

<a id="PLC-03"></a>
## PLC-03 · A grid's columns are as wide as their widest cell, a margin between cells and around them, never doubled; the diagram is its grid's size

In [ ]:
assert(text(b.child("cpu", "fpu")) === "\"[{\\\"kind\\\":\\\"place\\\",\\\"shape\\\":\\\"fpu\\\",\\\"order\\\":1,\\\"to\\\":[0,1]},{\\\"kind\\\":\\\"extents\\\",\\\"rows\\\":1,\\\"columns\\\":2}]\"");
assert(text(b.placed) === "\"[[\\\"overview\\\",\\\"frame\\\",[0,0],[0,0],[3.75,3.25],1],[\\\"overview/grid\\\",\\\"frame\\\",[0,0],[0,0],[3.75,3.25],1],[\\\"overview/grid/0,0\\\",\\\"frame\\\",[0.25,0.25],[0.25,0.25],[3.25,2.75],1],[\\\"cpu\\\",\\\"frame\\\",[0.5,0.5],[0.25,0.25],[2.75,2.25],0.5],[\\\"origin\\\",\\\"placement\\\",[1.875,1.75],[1.625,1.5],0],[\\\"n\\\",\\\"placement\\\",[1.875,0.75],[0,-1],0],[\\\"s\\\",\\\"placement\\\",[1.875,2.75],[0,1],180],[\\\"e\\\",\\\"placement\\\",[2.875,1.75],[1,0],90],[\\\"w\\\",\\\"placement\\\",[0.875,1.75],[-1,0],270],[\\\"cpu/grid\\\",\\\"frame\\\",[0.5,0.75],[0,0.25],[2.75,2],0.5],[\\\"cpu/grid/0,1\\\",\\\"frame\\\",[2.5,1],[2,0.25],[0.5,1.5],0.5],[\\\"fpu\\\",\\\"frame\\\",[2.75,1.75],[0.25,0.75],[0,0],0.25]]\"");
assert(text(b.points("cpu.fpu", [["n", 0, 4], ["s", 180, 4], ["e", 90, 2], ["w", 270, 2]])) === "\"[[\\\"overview\\\",\\\"frame\\\",[0,0],[0,0],[4.75,4.25],1],[\\\"overview/grid\\\",\\\"frame\\\",[0,0],[0,0],[4.75,4.25],1],[\\\"overview/grid/0,0\\\",\\\"frame\\\",[0.25,0.25],[0.25,0.25],[4.25,3.75],1],[\\\"cpu\\\",\\\"frame\\\",[0.5,0.5],[0.25,0.25],[3.75,3.25],0.5],[\\\"origin\\\",\\\"placement\\\",[2.375,2.25],[2.125,2],0],[\\\"n\\\",\\\"placement\\\",[2.375,1.25],[0,-1],0],[\\\"s\\\",\\\"placement\\\",[2.375,3.25],[0,1],180],[\\\"e\\\",\\\"placement\\\",[3.375,2.25],[1,0],90],[\\\"w\\\",\\\"placement\\\",[1.375,2.25],[-1,0],270],[\\\"cpu/grid\\\",\\\"frame\\\",[0.5,0.75],[0,0.25],[3.75,3],0.5],[\\\"cpu/grid/0,0\\\",\\\"frame\\\",[0.75,1],[0.25,0.25],[1.5,2.5],0.5],[\\\"alu\\\",\\\"frame\\\",[1,1.75],[0.25,0.75],[1,1],0.25],[\\\"origin\\\",\\\"placement\\\",[1.5,2.25],[0.75,1.25],0],[\\\"cpu/grid/0,1\\\",\\\"frame\\\",[2.5,1],[2,0.25],[1.5,2.5],0.5],[\\\"fpu\\\",\\\"frame\\\",[2.75,1.25],[0.25,0.25],[1,2],0.25],[\\\"origin\\\",\\\"placement\\\",[3.25,2.25],[0.75,1.25],0],[\\\"n\\\",\\\"placement\\\",[1.5,1.75],[0,-0.5],0],[\\\"s\\\",\\\"placement\\\",[1.5,2.75],[0,0.5],180],[\\\"e\\\",\\\"placement\\\",[2,2.25],[0.5,0],90],[\\\"w\\\",\\\"placement\\\",[1,2.25],[-0.5,0],270],[\\\"n\\\",\\\"placement\\\",[3.25,1.25],[0,-1],0],[\\\"s\\\",\\\"placement\\\",[3.25,3.25],[0,1],180],[\\\"e\\\",\\\"placement\\\",[3.75,2.25],[0.5,0],90],[\\\"w\\\",\\\"placement\\\",[2.75,2.25],[-0.5,0],270]]\"");

<a id="PLC-04"></a>
## PLC-04 · A child grows its container just enough to hold it, and the diagram with it

In [ ]:
assert(text(b.child("cpu", "mmu")) === "\"[{\\\"kind\\\":\\\"place\\\",\\\"shape\\\":\\\"mmu\\\",\\\"order\\\":2,\\\"to\\\":[0,2]},{\\\"kind\\\":\\\"extents\\\",\\\"rows\\\":1,\\\"columns\\\":3}]\"");
assert(text(b.placed) === "\"[[\\\"overview\\\",\\\"frame\\\",[0,0],[0,0],[5.5,4.25],1],[\\\"overview/grid\\\",\\\"frame\\\",[0,0],[0,0],[5.5,4.25],1],[\\\"overview/grid/0,0\\\",\\\"frame\\\",[0.25,0.25],[0.25,0.25],[5,3.75],1],[\\\"cpu\\\",\\\"frame\\\",[0.5,0.5],[0.25,0.25],[4.5,3.25],0.5],[\\\"origin\\\",\\\"placement\\\",[2.75,2.25],[2.5,2],0],[\\\"n\\\",\\\"placement\\\",[2.75,1.25],[0,-1],0],[\\\"s\\\",\\\"placement\\\",[2.75,3.25],[0,1],180],[\\\"e\\\",\\\"placement\\\",[3.75,2.25],[1,0],90],[\\\"w\\\",\\\"placement\\\",[1.75,2.25],[-1,0],270],[\\\"cpu/grid\\\",\\\"frame\\\",[0.5,0.75],[0,0.25],[4.5,3],0.5],[\\\"cpu/grid/0,2\\\",\\\"frame\\\",[4.25,1],[3.75,0.25],[0.5,2.5],0.5],[\\\"mmu\\\",\\\"frame\\\",[4.5,2.25],[0.25,1.25],[0,0],0.25]]\"");
assert(text(b.points("cpu.mmu", SQUARE)) === "\"[[\\\"overview\\\",\\\"frame\\\",[0,0],[0,0],[6.5,4.25],1],[\\\"overview/grid\\\",\\\"frame\\\",[0,0],[0,0],[6.5,4.25],1],[\\\"overview/grid/0,0\\\",\\\"frame\\\",[0.25,0.25],[0.25,0.25],[6,3.75],1],[\\\"cpu\\\",\\\"frame\\\",[0.5,0.5],[0.25,0.25],[5.5,3.25],0.5],[\\\"origin\\\",\\\"placement\\\",[3.25,2.25],[3,2],0],[\\\"n\\\",\\\"placement\\\",[3.25,1.25],[0,-1],0],[\\\"s\\\",\\\"placement\\\",[3.25,3.25],[0,1],180],[\\\"e\\\",\\\"placement\\\",[4.25,2.25],[1,0],90],[\\\"w\\\",\\\"placement\\\",[2.25,2.25],[-1,0],270],[\\\"cpu/grid\\\",\\\"frame\\\",[0.5,0.75],[0,0.25],[5.5,3],0.5],[\\\"cpu/grid/0,2\\\",\\\"frame\\\",[4.25,1],[3.75,0.25],[1.5,2.5],0.5],[\\\"mmu\\\",\\\"frame\\\",[4.5,1.75],[0.25,0.75],[1,1],0.25],[\\\"origin\\\",\\\"placement\\\",[5,2.25],[0.75,1.25],0],[\\\"n\\\",\\\"placement\\\",[5,1.75],[0,-0.5],0],[\\\"s\\\",\\\"placement\\\",[5,2.75],[0,0.5],180],[\\\"e\\\",\\\"placement\\\",[5.5,2.25],[0.5,0],90],[\\\"w\\\",\\\"placement\\\",[4.5,2.25],[-0.5,0],270]]\"");

<a id="PLC-05"></a>
## PLC-05 · A change that moves nothing writes nothing; a removal shrinks what grew

In [ ]:
assert(text(b.spec("cpu", "fpu", "alu", 270)) === "\"[{\\\"kind\\\":\\\"rule\\\",\\\"from_shape\\\":\\\"fpu\\\",\\\"to_shape\\\":\\\"alu\\\",\\\"degrees\\\":270,\\\"status\\\":\\\"held\\\"}]\"");
assert(text(b.placed) === "\"[]\"");
assert(text(b.unchild("cpu", "mmu")) === "\"[{\\\"kind\\\":\\\"leave\\\",\\\"shape\\\":\\\"mmu\\\",\\\"from\\\":[0,2]},{\\\"kind\\\":\\\"extents\\\",\\\"rows\\\":1,\\\"columns\\\":2},{\\\"kind\\\":\\\"place\\\",\\\"shape\\\":\\\"mmu\\\",\\\"order\\\":1,\\\"to\\\":[0,1]},{\\\"kind\\\":\\\"extents\\\",\\\"rows\\\":1,\\\"columns\\\":2}]\"");
assert(text(b.placed) === "\"[[\\\"overview\\\",\\\"frame\\\",[0,0],[0,0],[7.5,4.25],1],[\\\"overview/grid\\\",\\\"frame\\\",[0,0],[0,0],[7.5,4.25],1],[\\\"overview/grid/0,0\\\",\\\"frame\\\",[0.25,0.25],[0.25,0.25],[4.25,3.75],1],[\\\"cpu\\\",\\\"frame\\\",[0.5,0.5],[0.25,0.25],[3.75,3.25],0.5],[\\\"origin\\\",\\\"placement\\\",[2.375,2.25],[2.125,2],0],[\\\"overview/grid/0,1\\\",\\\"frame\\\",[4.75,0.25],[4.75,0.25],[2.5,3.75],1],[\\\"mmu\\\",\\\"frame\\\",[5,1.125],[0.25,0.875],[2,2],0.5],[\\\"origin\\\",\\\"placement\\\",[6,2.125],[1.25,1.875],0],[\\\"n\\\",\\\"placement\\\",[2.375,1.25],[0,-1],0],[\\\"s\\\",\\\"placement\\\",[2.375,3.25],[0,1],180],[\\\"e\\\",\\\"placement\\\",[3.375,2.25],[1,0],90],[\\\"w\\\",\\\"placement\\\",[1.375,2.25],[-1,0],270],[\\\"cpu/grid\\\",\\\"frame\\\",[0.5,0.75],[0,0.25],[3.75,3],0.5],[\\\"n\\\",\\\"placement\\\",[6,1.125],[0,-1],0],[\\\"s\\\",\\\"placement\\\",[6,3.125],[0,1],180],[\\\"e\\\",\\\"placement\\\",[7,2.125],[1,0],90],[\\\"w\\\",\\\"placement\\\",[5,2.125],[-1,0],270]]\"");

<a id="PLC-06"></a>
## PLC-06 · A shared cell stacks its shapes along its grid's axis, a margin between them

In [ ]:
assert(text(b.spec("cpu", "alu", "fpu", 90)) === "\"[{\\\"kind\\\":\\\"rule\\\",\\\"from_shape\\\":\\\"alu\\\",\\\"to_shape\\\":\\\"fpu\\\",\\\"degrees\\\":90,\\\"status\\\":\\\"held\\\"}]\"");
assert(text(b.child("cpu", "dsp")) === "\"[{\\\"kind\\\":\\\"place\\\",\\\"shape\\\":\\\"dsp\\\",\\\"order\\\":3,\\\"to\\\":[0,2]},{\\\"kind\\\":\\\"extents\\\",\\\"rows\\\":1,\\\"columns\\\":3}]\"");
b.points("cpu.dsp", SQUARE);
assert(text(b.spec("cpu", "alu", "dsp", 90)) === "\"[{\\\"kind\\\":\\\"move\\\",\\\"shape\\\":\\\"dsp\\\",\\\"from\\\":[0,2],\\\"to\\\":[0,1]},{\\\"kind\\\":\\\"rule\\\",\\\"from_shape\\\":\\\"alu\\\",\\\"to_shape\\\":\\\"dsp\\\",\\\"degrees\\\":90,\\\"status\\\":\\\"met\\\"},{\\\"kind\\\":\\\"extents\\\",\\\"rows\\\":1,\\\"columns\\\":2}]\"");
assert(text(b.placed) === "\"[[\\\"overview\\\",\\\"frame\\\",[0,0],[0,0],[7.5,5.5],1],[\\\"overview/grid\\\",\\\"frame\\\",[0,0],[0,0],[7.5,5.5],1],[\\\"overview/grid/0,0\\\",\\\"frame\\\",[0.25,0.25],[0.25,0.25],[4.25,5],1],[\\\"cpu\\\",\\\"frame\\\",[0.5,0.5],[0.25,0.25],[3.75,4.5],0.5],[\\\"origin\\\",\\\"placement\\\",[2.375,2.875],[2.125,2.625],0],[\\\"overview/grid/0,1\\\",\\\"frame\\\",[4.75,0.25],[4.75,0.25],[2.5,5],1],[\\\"mmu\\\",\\\"frame\\\",[5,1.75],[0.25,1.5],[2,2],0.5],[\\\"origin\\\",\\\"placement\\\",[6,2.75],[1.25,2.5],0],[\\\"n\\\",\\\"placement\\\",[2.375,1.875],[0,-1],0],[\\\"s\\\",\\\"placement\\\",[2.375,3.875],[0,1],180],[\\\"e\\\",\\\"placement\\\",[3.375,2.875],[1,0],90],[\\\"w\\\",\\\"placement\\\",[1.375,2.875],[-1,0],270],[\\\"cpu/grid\\\",\\\"frame\\\",[0.5,0.75],[0,0.25],[3.75,4.25],0.5],[\\\"cpu/grid/0,0\\\",\\\"frame\\\",[0.75,1],[0.25,0.25],[1.5,3.75],0.5],[\\\"alu\\\",\\\"frame\\\",[1,2.375],[0.25,1.375],[1,1],0.25],[\\\"origin\\\",\\\"placement\\\",[1.5,2.875],[0.75,1.875],0],[\\\"cpu/grid/0,1\\\",\\\"frame\\\",[2.5,1],[2,0.25],[1.5,3.75],0.5],[\\\"dsp\\\",\\\"frame\\\",[2.75,3.5],[0.25,2.5],[1,1],0.25],[\\\"origin\\\",\\\"placement\\\",[3.25,4],[0.75,3],0],[\\\"n\\\",\\\"placement\\\",[1.5,2.375],[0,-0.5],0],[\\\"s\\\",\\\"placement\\\",[1.5,3.375],[0,0.5],180],[\\\"e\\\",\\\"placement\\\",[2,2.875],[0.5,0],90],[\\\"w\\\",\\\"placement\\\",[1,2.875],[-0.5,0],270],[\\\"n\\\",\\\"placement\\\",[6,1.75],[0,-1],0],[\\\"s\\\",\\\"placement\\\",[6,3.75],[0,1],180],[\\\"e\\\",\\\"placement\\\",[7,2.75],[1,0],90],[\\\"w\\\",\\\"placement\\\",[5,2.75],[-1,0],270],[\\\"n\\\",\\\"placement\\\",[3.25,3.5],[0,-0.5],0],[\\\"s\\\",\\\"placement\\\",[3.25,4.5],[0,0.5],180],[\\\"e\\\",\\\"placement\\\",[3.75,4],[0.5,0],90],[\\\"w\\\",\\\"placement\\\",[2.75,4],[-0.5,0],270]]\"");

<a id="PLC-07"></a>
## PLC-07 · Turning a container's cells turns the points of the shapes they hold

In [ ]:
assert(text(b.turn("cpu", 90)) === "\"[{\\\"kind\\\":\\\"grid\\\",\\\"container\\\":\\\"cpu\\\",\\\"span\\\":3,\\\"grows\\\":\\\"rows\\\",\\\"cell_rotation\\\":90}]\"");
assert(text(b.placed) === "\"[[\\\"overview\\\",\\\"frame\\\",[0,0],[0,0],[8.5,4.5],1],[\\\"overview/grid\\\",\\\"frame\\\",[0,0],[0,0],[8.5,4.5],1],[\\\"overview/grid/0,0\\\",\\\"frame\\\",[0.25,0.25],[0.25,0.25],[5.25,4],1],[\\\"cpu\\\",\\\"frame\\\",[0.5,0.5],[0.25,0.25],[4.75,3.5],0.5],[\\\"origin\\\",\\\"placement\\\",[2.875,2.375],[2.625,2.125],0],[\\\"overview/grid/0,1\\\",\\\"frame\\\",[5.75,0.25],[5.75,0.25],[2.5,4],1],[\\\"mmu\\\",\\\"frame\\\",[6,1.25],[0.25,1],[2,2],0.5],[\\\"origin\\\",\\\"placement\\\",[7,2.25],[1.25,2],0],[\\\"n\\\",\\\"placement\\\",[2.875,1.375],[0,-1],0],[\\\"s\\\",\\\"placement\\\",[2.875,3.375],[0,1],180],[\\\"e\\\",\\\"placement\\\",[3.875,2.375],[1,0],90],[\\\"w\\\",\\\"placement\\\",[1.875,2.375],[-1,0],270],[\\\"cpu/grid\\\",\\\"frame\\\",[0.5,0.75],[0,0.25],[4.75,3.25],0.5],[\\\"cpu/grid/0,0\\\",\\\"frame\\\",[0.75,1],[0.25,0.25],[1.5,2.75],0.5],[\\\"alu\\\",\\\"frame\\\",[1,1.875],[0.25,0.875],[1,1],0.25],[\\\"origin\\\",\\\"placement\\\",[1.5,2.375],[0.75,1.375],90],[\\\"cpu/grid/0,1\\\",\\\"frame\\\",[2.5,1],[2,0.25],[2.5,2.75],0.5],[\\\"fpu\\\",\\\"frame\\\",[2.75,1.25],[0.25,0.25],[2,1],0.25],[\\\"origin\\\",\\\"placement\\\",[3.75,1.75],[1.25,0.75],90],[\\\"dsp\\\",\\\"frame\\\",[3.25,2.5],[0.75,1.5],[1,1],0.25],[\\\"origin\\\",\\\"placement\\\",[3.75,3],[1.25,2],90],[\\\"n\\\",\\\"placement\\\",[2,2.375],[0.5,0],90],[\\\"s\\\",\\\"placement\\\",[1,2.375],[-0.5,0],270],[\\\"e\\\",\\\"placement\\\",[1.5,2.875],[0,0.5],180],[\\\"w\\\",\\\"placement\\\",[1.5,1.875],[0,-0.5],0],[\\\"n\\\",\\\"placement\\\",[4.75,1.75],[1,0],90],[\\\"s\\\",\\\"placement\\\",[2.75,1.75],[-1,0],270],[\\\"e\\\",\\\"placement\\\",[3.75,2.25],[0,0.5],180],[\\\"w\\\",\\\"placement\\\",[3.75,1.25],[0,-0.5],0],[\\\"n\\\",\\\"placement\\\",[7,1.25],[0,-1],0],[\\\"s\\\",\\\"placement\\\",[7,3.25],[0,1],180],[\\\"e\\\",\\\"placement\\\",[8,2.25],[1,0],90],[\\\"w\\\",\\\"placement\\\",[6,2.25],[-1,0],270],[\\\"n\\\",\\\"placement\\\",[4.25,3],[0.5,0],90],[\\\"s\\\",\\\"placement\\\",[3.25,3],[-0.5,0],270],[\\\"e\\\",\\\"placement\\\",[3.75,3.5],[0,0.5],180],[\\\"w\\\",\\\"placement\\\",[3.75,2.5],[0,-0.5],0]]\"");

<a id="PLC-08"></a>
## PLC-08 · A shape is at least its minimum size, in units of its level

In [ ]:
const small = new Bench(true, true); small.tune({ unit_length: 1, scaling_factor: 0.5, minimum_size: 3 });
assert(text(small.child("", "dot")) === "\"[{\\\"kind\\\":\\\"grid\\\",\\\"container\\\":\\\"overview\\\",\\\"span\\\":3,\\\"grows\\\":\\\"rows\\\",\\\"cell_rotation\\\":0},{\\\"kind\\\":\\\"place\\\",\\\"shape\\\":\\\"dot\\\",\\\"order\\\":0,\\\"to\\\":[0,0]},{\\\"kind\\\":\\\"extents\\\",\\\"rows\\\":1,\\\"columns\\\":1}]\"");
assert(text(small.placed) === "\"[[\\\"overview\\\",\\\"frame\\\",[0,0],[0,0],[1.5,1.5],1],[\\\"overview/grid\\\",\\\"frame\\\",[0,0],[0,0],[1.5,1.5],1],[\\\"overview/grid/0,0\\\",\\\"frame\\\",[0,0],[0,0],[1.5,1.5],1],[\\\"dot\\\",\\\"frame\\\",[0,0],[0,0],[1.5,1.5],0.5]]\"");

<a id="PLC-09"></a>
## PLC-09 · A grid that grows columns stacks a shared cell across, and lays its cells out column by column

In [ ]:
const col = new Bench(true, true); col.tune({ unit_length: 1, scaling_factor: 0.5, margin: 0.25 });
col.child("", "bus"); col.layout("bus", 2n, "columns");
for (const n of ["p", "q", "r"]) { col.child("bus", n); col.points(`bus.${n}`, SQUARE); }
assert(text(col.spec("bus", "p", "q", 180)) === "\"[{\\\"kind\\\":\\\"rule\\\",\\\"from_shape\\\":\\\"p\\\",\\\"to_shape\\\":\\\"q\\\",\\\"degrees\\\":180,\\\"status\\\":\\\"held\\\"}]\"");
assert(text(col.spec("bus", "p", "r", 180)) === "\"[{\\\"kind\\\":\\\"move\\\",\\\"shape\\\":\\\"r\\\",\\\"from\\\":[0,1],\\\"to\\\":[1,0]},{\\\"kind\\\":\\\"rule\\\",\\\"from_shape\\\":\\\"p\\\",\\\"to_shape\\\":\\\"r\\\",\\\"degrees\\\":180,\\\"status\\\":\\\"met\\\"},{\\\"kind\\\":\\\"extents\\\",\\\"rows\\\":2,\\\"columns\\\":1}]\"");
assert(text(col.placed) === "\"[[\\\"overview\\\",\\\"frame\\\",[0,0],[0,0],[4.25,5],1],[\\\"overview/grid\\\",\\\"frame\\\",[0,0],[0,0],[4.25,5],1],[\\\"overview/grid/0,0\\\",\\\"frame\\\",[0.25,0.25],[0.25,0.25],[3.75,4.5],1],[\\\"bus\\\",\\\"frame\\\",[0.5,0.5],[0.25,0.25],[3.25,4],0.5],[\\\"bus/grid\\\",\\\"frame\\\",[0.5,0.75],[0,0.25],[3.25,3.75],0.5],[\\\"bus/grid/0,0\\\",\\\"frame\\\",[0.75,1],[0.25,0.25],[2.75,1.5],0.5],[\\\"p\\\",\\\"frame\\\",[1.625,1.25],[0.875,0.25],[1,1],0.25],[\\\"origin\\\",\\\"placement\\\",[2.125,1.75],[1.375,0.75],0],[\\\"bus/grid/1,0\\\",\\\"frame\\\",[0.75,2.75],[0.25,2],[2.75,1.5],0.5],[\\\"r\\\",\\\"frame\\\",[2.25,3],[1.5,0.25],[1,1],0.25],[\\\"origin\\\",\\\"placement\\\",[2.75,3.5],[2,0.75],0],[\\\"n\\\",\\\"placement\\\",[2.125,1.25],[0,-0.5],0],[\\\"s\\\",\\\"placement\\\",[2.125,2.25],[0,0.5],180],[\\\"e\\\",\\\"placement\\\",[2.625,1.75],[0.5,0],90],[\\\"w\\\",\\\"placement\\\",[1.625,1.75],[-0.5,0],270],[\\\"n\\\",\\\"placement\\\",[2.75,3],[0,-0.5],0],[\\\"s\\\",\\\"placement\\\",[2.75,4],[0,0.5],180],[\\\"e\\\",\\\"placement\\\",[3.25,3.5],[0.5,0],90],[\\\"w\\\",\\\"placement\\\",[2.25,3.5],[-0.5,0],270]]\"");

<a id="PLC-10"></a>
## PLC-10 · A container whose last child goes lays out an empty grid

In [ ]:
assert(text(col.unchild("bus", "p", "q", "r")) === "\"[{\\\"kind\\\":\\\"leave\\\",\\\"shape\\\":\\\"p\\\",\\\"from\\\":[0,0]},{\\\"kind\\\":\\\"move\\\",\\\"shape\\\":\\\"q\\\",\\\"from\\\":[1,0],\\\"to\\\":[0,0]},{\\\"kind\\\":\\\"move\\\",\\\"shape\\\":\\\"r\\\",\\\"from\\\":[1,0],\\\"to\\\":[0,0]},{\\\"kind\\\":\\\"rule\\\",\\\"from_shape\\\":\\\"p\\\",\\\"to_shape\\\":\\\"q\\\",\\\"degrees\\\":180,\\\"status\\\":\\\"dropped\\\"},{\\\"kind\\\":\\\"rule\\\",\\\"from_shape\\\":\\\"p\\\",\\\"to_shape\\\":\\\"r\\\",\\\"degrees\\\":180,\\\"status\\\":\\\"dropped\\\"},{\\\"kind\\\":\\\"extents\\\",\\\"rows\\\":1,\\\"columns\\\":1},{\\\"kind\\\":\\\"leave\\\",\\\"shape\\\":\\\"q\\\",\\\"from\\\":[0,0]},{\\\"kind\\\":\\\"leave\\\",\\\"shape\\\":\\\"r\\\",\\\"from\\\":[0,0]},{\\\"kind\\\":\\\"extents\\\",\\\"rows\\\":0,\\\"columns\\\":0},{\\\"kind\\\":\\\"place\\\",\\\"shape\\\":\\\"p\\\",\\\"order\\\":1,\\\"to\\\":[0,1]},{\\\"kind\\\":\\\"extents\\\",\\\"rows\\\":1,\\\"columns\\\":2},{\\\"kind\\\":\\\"place\\\",\\\"shape\\\":\\\"q\\\",\\\"order\\\":2,\\\"to\\\":[0,2]},{\\\"kind\\\":\\\"extents\\\",\\\"rows\\\":1,\\\"columns\\\":3},{\\\"kind\\\":\\\"place\\\",\\\"shape\\\":\\\"r\\\",\\\"order\\\":3,\\\"to\\\":[1,0]},{\\\"kind\\\":\\\"extents\\\",\\\"rows\\\":2,\\\"columns\\\":3}]\"");
assert(text(col.placed) === "\"[[\\\"overview\\\",\\\"frame\\\",[0,0],[0,0],[8.5,5.75],1],[\\\"overview/grid\\\",\\\"frame\\\",[0,0],[0,0],[8.5,5.75],1],[\\\"overview/grid/0,0\\\",\\\"frame\\\",[0.25,0.25],[0.25,0.25],[2.5,2.5],1],[\\\"bus\\\",\\\"frame\\\",[1.5,1.375],[1.25,1.125],[0,0.25],0.5],[\\\"overview/grid/0,1\\\",\\\"frame\\\",[3,0.25],[3,0.25],[2.5,2.5],1],[\\\"p\\\",\\\"frame\\\",[3.25,0.5],[0.25,0.25],[2,2],0.5],[\\\"origin\\\",\\\"placement\\\",[4.25,1.5],[1.25,1.25],0],[\\\"overview/grid/0,2\\\",\\\"frame\\\",[5.75,0.25],[5.75,0.25],[2.5,2.5],1],[\\\"q\\\",\\\"frame\\\",[6,0.5],[0.25,0.25],[2,2],0.5],[\\\"origin\\\",\\\"placement\\\",[7,1.5],[1.25,1.25],0],[\\\"overview/grid/1,0\\\",\\\"frame\\\",[0.25,3],[0.25,3],[2.5,2.5],1],[\\\"r\\\",\\\"frame\\\",[0.5,3.25],[0.25,0.25],[2,2],0.5],[\\\"origin\\\",\\\"placement\\\",[1.5,4.25],[1.25,1.25],0],[\\\"bus/grid\\\",\\\"frame\\\",[1.5,1.625],[0,0.25],[0,0],0.5],[\\\"n\\\",\\\"placement\\\",[4.25,0.5],[0,-1],0],[\\\"s\\\",\\\"placement\\\",[4.25,2.5],[0,1],180],[\\\"e\\\",\\\"placement\\\",[5.25,1.5],[1,0],90],[\\\"w\\\",\\\"placement\\\",[3.25,1.5],[-1,0],270],[\\\"n\\\",\\\"placement\\\",[7,0.5],[0,-1],0],[\\\"s\\\",\\\"placement\\\",[7,2.5],[0,1],180],[\\\"e\\\",\\\"placement\\\",[8,1.5],[1,0],90],[\\\"w\\\",\\\"placement\\\",[6,1.5],[-1,0],270],[\\\"n\\\",\\\"placement\\\",[1.5,3.25],[0,-1],0],[\\\"s\\\",\\\"placement\\\",[1.5,5.25],[0,1],180],[\\\"e\\\",\\\"placement\\\",[2.5,4.25],[1,0],90],[\\\"w\\\",\\\"placement\\\",[0.5,4.25],[-1,0],270]]\"");

<a id="PLC-11"></a>
## PLC-11 · A point no vector from its origin reaches lies at its origin, and a vector with no length reaches no farther

In [ ]:
const odd = new Bench(true, true);
odd.child("", "box");
const L = odd.source.Logical;
const s = "s_box";
odd.step("points", ["box", []], ["box"], { origin: L.OfPoint, lost: L.OfPoint, short: L.OfPoint, v: L.OfVector },
  (_: any) => _[s].owns({ origin: (e: any) => e.owned(_.origin) }), (_: any) => _.origin.origin_of([_[s]]),
  (_: any) => _[s].owns({ lost: (e: any) => e.owned(_.lost) }), (_: any) => _.lost.border_of([_[s]]),
  (_: any) => _[s].owns({ short: (e: any) => e.owned(_.short) }), (_: any) => _.short.border_of([_[s]]),
  (_: any) => _[s].owns({ "to short": (e: any) => e.owned(_.v) }),
  (_: any) => _.v.start([_.origin]).end([_.short]).rotation((r: any) => r.degrees(90)));
assert(text(odd.placed) === "\"[[\\\"origin\\\",\\\"placement\\\",[0,0],[0,0],0],[\\\"lost\\\",\\\"placement\\\",[0,0],[0,0],0],[\\\"short\\\",\\\"placement\\\",[0,0],[0,0],90]]\"");

<a id="PLC-12"></a>
## PLC-12 · What placement writes is a mutation: an upsert of each frame and placement it changed, found by its owner's path, which the store runs

In [ ]:
const again = new Placement.Layout(b.overview).follow([b.overview]);
assert(text(again.as_schema().name) === "\"Mutations.OfTransaction\"");
b.store.transaction(again);
assert(text(b.afresh()) === "[]");

<a id="PLC-13"></a>
## PLC-13 · A required constraint that conflicts with those held is ignored and queued, and holds, in its order, once what it conflicts with goes

In [ ]:
const set = new Constraints.ConstraintSet(new Kiwi.KiwiEngine());
const is = (v: string, k: number): Constraints.Constraint => ({ terms: [[1, v]], constant: -k, op: "==", strength: "required" });
set.change("a", is("x", 1)); set.change("b", is("x", 2)); set.change("c", is("x", 3));
set.change("y", { terms: [[1, "y"]], constant: -5, op: "<=", strength: "required" });
set.change("y small", { terms: [[1, "y"]], constant: 0, op: "==", strength: "weak" });
set.solve();
assert(text([set.value("x"), set.ignored(), set.value("y"), set.value("never")]) === "[1,[\"b\",\"c\"],0,0]");
set.change("a", is("x", 1)); // the same: nothing changes
set.change("a", null);
set.change("y small", null);
set.change("y big", { terms: [[1, "y"]], constant: -9, op: "==", strength: "weak" });
set.solve();
assert(text([set.value("x"), set.ignored(), set.value("y")]) === "[2,[\"c\"],5]");
assert(text(b.driver.placement(b.overview)?.ignored()) === "[]");
assert(b.driver.placement(b.overview)?.follow([b.store.singleton("Logical.Diagrams")]) === null); // nothing placed

<a id="PLC-14"></a>
## PLC-14 · Following mutation by mutation ends as placing afresh does, through a seeded sequence of every kind of mutation

In [ ]:
const r = new Bench(true, true);
r.tune({ unit_length: 1, scaling_factor: 0.5, margin: 0.25, minimum_size: 1 });
let seed = 2026;
const pick = (n: number): number => { seed = (seed * 48271) % 2147483647; return Math.floor(seed / 65536) % n; }; // exact in doubles
const kids = new Map<string, string[]>([["", []]]); // each container's children, by handle
const kinds: string[] = [];
const drift: number[] = [];
for (let i = 0; i < 150; i++) {
  const boxes = [...kids.keys()];
  const box = boxes[pick(boxes.length)] as string;
  const here = kids.get(box) as string[];
  const kind = ["child", "child", "points", "spec", "unspec", "unchild", "turn", "tune", "move"][pick(9)] as string;
  if (kind === "child") {
    const handle = box ? `${box}.n${i}` : `n${i}`;
    r.child(box, `n${i}`);
    here.push(handle);
    kids.set(handle, []);
  } else if (kind === "points" && box) {
    const k = 1 + pick(3);
    r.points(box, [["n", 0, k], ["s", 180, 1 + pick(3)], ["e", 90, k], ["w", 270, 1]]);
  } else if ((kind === "spec" || kind === "unspec") && here.length > 1) {
    const [a, c] = [here[pick(here.length)] as string, here[pick(here.length)] as string];
    if (a === c) continue;
    if (kind === "spec") r.spec("", a, c, 90 * pick(4)); // by handle
    else r.unspec("", a, c);
  } else if (kind === "unchild" && box && here.length) { // to the diagram's top level, keeping its handle
    const gone = here.splice(pick(here.length), 1)[0] as string;
    r.move(gone, "");
    (kids.get("") as string[]).push(gone);
  } else if (kind === "move" && box) { // a top-level shape into a container, never into itself
    const top = kids.get("") as string[];
    if (!top.length) continue;
    const handle = top[pick(top.length)] as string;
    let inside = false;
    for (let c = r.shape(box); !inside && c.as_schema().name !== "Logical.OfDiagram"; c = Logical.container_of(c)) {
      inside = Logical.name_of(c) === handle;
    }
    if (inside) continue;
    r.move(handle, box);
    top.splice(top.indexOf(handle), 1);
    here.push(handle);
  } else if (kind === "turn" && box && here.length) {
    r.turn(box, 90 * pick(4));
  } else if (kind === "tune") {
    r.tune({ margin: 0.25 * pick(3) });
  } else continue;
  kinds.push(kind);
  if (r.afresh().length) drift.push(i);
}
assert(text([...new Set(kinds)].sort()) === "[\"child\",\"move\",\"points\",\"spec\",\"tune\",\"turn\",\"unchild\",\"unspec\"]");
assert(text(kinds.length) === "101");
assert(text(drift) === "[]");

<a id="PLC-15"></a>
## PLC-15 · A container keeps a band for its title, `title_size` units of its level, between its top border and its children; none where it is 0

In [ ]:
const titled = new Bench(true, true); titled.tune({ unit_length: 1, scaling_factor: 0.5, margin: 0.25, title_size: 1 });
titled.child("", "box"); titled.child("box", "x");
assert(text(titled.points("box.x", SQUARE)) === "\"[[\\\"overview\\\",\\\"frame\\\",[0,0],[0,0],[3,3.5],1],[\\\"overview/grid\\\",\\\"frame\\\",[0,0],[0,0],[3,3.5],1],[\\\"overview/grid/0,0\\\",\\\"frame\\\",[0.25,0.25],[0.25,0.25],[2.5,3],1],[\\\"box\\\",\\\"frame\\\",[0.5,0.5],[0.25,0.25],[2,2.5],0.5],[\\\"box/grid\\\",\\\"frame\\\",[0.5,1],[0,0.5],[2,2],0.5],[\\\"box/grid/0,0\\\",\\\"frame\\\",[0.75,1.25],[0.25,0.25],[1.5,1.5],0.5],[\\\"x\\\",\\\"frame\\\",[1,1.5],[0.25,0.25],[1,1],0.25],[\\\"origin\\\",\\\"placement\\\",[1.5,2],[0.75,0.75],0],[\\\"n\\\",\\\"placement\\\",[1.5,1.5],[0,-0.5],0],[\\\"s\\\",\\\"placement\\\",[1.5,2.5],[0,0.5],180],[\\\"e\\\",\\\"placement\\\",[2,2],[0.5,0],90],[\\\"w\\\",\\\"placement\\\",[1,2],[-0.5,0],270]]\"");
assert(text(titled.tune({ title_size: 0 })) === "\"[]\"");
assert(text(titled.placed) === "\"[[\\\"overview\\\",\\\"frame\\\",[0,0],[0,0],[3,3],1],[\\\"overview/grid\\\",\\\"frame\\\",[0,0],[0,0],[3,3],1],[\\\"overview/grid/0,0\\\",\\\"frame\\\",[0.25,0.25],[0.25,0.25],[2.5,2.5],1],[\\\"box\\\",\\\"frame\\\",[0.5,0.5],[0.25,0.25],[2,2],0.5],[\\\"box/grid\\\",\\\"frame\\\",[0.5,0.5],[0,0],[2,2],0.5],[\\\"box/grid/0,0\\\",\\\"frame\\\",[0.75,0.75],[0.25,0.25],[1.5,1.5],0.5],[\\\"x\\\",\\\"frame\\\",[1,1],[0.25,0.25],[1,1],0.25],[\\\"origin\\\",\\\"placement\\\",[1.5,1.5],[0.75,0.75],0],[\\\"n\\\",\\\"placement\\\",[1.5,1],[0,-0.5],0],[\\\"s\\\",\\\"placement\\\",[1.5,2],[0,0.5],180],[\\\"e\\\",\\\"placement\\\",[2,1.5],[0.5,0],90],[\\\"w\\\",\\\"placement\\\",[1,1.5],[-0.5,0],270]]\"");

<a id="PLC-16"></a>
## PLC-16 · The journal, replayed in a fresh store, places everything the same

In [ ]:
assert(text(b.replay()) === "[]");
assert(text(small.replay()) === "[]");
assert(text(col.replay()) === "[]");
assert(text(odd.replay()) === "[]");
assert(text(r.replay()) === "[]");
assert(text(titled.replay()) === "[]");